# 02 DAG and Discovery Review

Review-only notebook for the hand-built DAG and the reduced discovery variable list. This notebook does not run causal discovery.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import yaml
from IPython.display import Image, display

from oulad_causal.config import FIGURES_DIR, PROCESSED_DATA_DIR
from oulad_causal.dag import (
    DAG_FIGURE_PATH,
    DAG_VARIABLE_AVAILABILITY_PATH,
    DAG_YAML_PATH,
    discovery_variable_list,
    primary_dag_spec,
    recommended_baseline_adjustment_set,
)

## Hand-Built DAG

The saved DAG is the domain-informed baseline for identification. Discovery output, when added later, should be compared against this graph rather than treated as definitive truth.

In [ ]:
if DAG_YAML_PATH.exists():
    with DAG_YAML_PATH.open(encoding="utf-8") as handle:
        dag_spec = yaml.safe_load(handle)
else:
    dag_spec = primary_dag_spec()

pd.DataFrame(dag_spec["nodes"])[["id", "label", "role", "observed", "cohort_columns"]]

In [ ]:
if DAG_FIGURE_PATH.exists():
    display(Image(filename=str(DAG_FIGURE_PATH)))
else:
    print(f"DAG figure not found yet: {DAG_FIGURE_PATH}")

## Primary Adjustment Set

In [ ]:
pd.Series(recommended_baseline_adjustment_set(), name="recommended_baseline_adjustment_column").to_frame()

## Cohort Metadata and Variable Availability

In [ ]:
summary_path = PROCESSED_DATA_DIR / "cohort_summary.json"
with summary_path.open(encoding="utf-8") as handle:
    cohort_summary = json.load(handle)

cohort_summary["cohort_size"], cohort_summary["primary_window_days"], cohort_summary["exclusion_counts"]

In [ ]:
cohort_path = PROCESSED_DATA_DIR / "oulad_analytic_cohort.parquet"
cohort_columns = set(pd.read_parquet(cohort_path).columns)

availability = []
for node in dag_spec["nodes"]:
    columns = node["cohort_columns"]
    missing = [column for column in columns if column not in cohort_columns]
    availability.append(
        {
            "source": "dag_node",
            "name": node["id"],
            "role": node["role"],
            "columns": ", ".join(columns),
            "available": bool(columns) and not missing,
            "missing_columns": ", ".join(missing),
        }
    )

for column in discovery_variable_list():
    availability.append(
        {
            "source": "discovery_variable",
            "name": column,
            "role": "prepared_for_later_review",
            "columns": column,
            "available": column in cohort_columns,
            "missing_columns": "" if column in cohort_columns else column,
        }
    )

pd.DataFrame(availability)

In [ ]:
if DAG_VARIABLE_AVAILABILITY_PATH.exists():
    display(pd.read_csv(DAG_VARIABLE_AVAILABILITY_PATH))
else:
    print(f"Saved availability table not found yet: {DAG_VARIABLE_AVAILABILITY_PATH}")

## Prepared Discovery Variables

Discovery has not been run. The list below is only the reduced candidate input list for later PC/FCI/GES-style review after encoding and method choices are documented.

In [ ]:
pd.Series(discovery_variable_list(), name="prepared_discovery_variable").to_frame()